In [2]:
import os
print(os.getcwd())

/Users/danielseymour/Developer/EC331-Thesis-Data/data-preparation


In [6]:
import pandas as pd
import os

# Define the file path
file_path = '/Users/danielseymour/Downloads/MMSDM_2023_11-2/MMSDM_Historical_Data_SQLLoader/DATA/PUBLIC_DVD_BIDPEROFFER1_202311010000.CSV'

# Verify the file exists
if not os.path.exists(file_path):
    print(f"File not found: {file_path}")
else:
    # Skip the first row and use the second row as headers
    df = pd.read_csv(file_path, skiprows=1)
    
    # Print basic information about the dataset
    print(f"Total records: {len(df)}")
    print(f"Columns: {df.columns.tolist()}")
    
    # Check if BIDTYPE column exists
    if 'BIDTYPE' in df.columns:
        # Process in chunks to avoid memory issues
        chunk_size = 100000  # Adjust based on your available memory
        total_filtered_records = 0
        output_file = 'filtered_raise1sec.csv'
        
        # Process first chunk and write headers
        first_chunk = True
        
        for chunk in pd.read_csv(file_path, skiprows=1, chunksize=chunk_size):
            # Filter for BIDTYPE = RAISE1SEC
            filtered_chunk = chunk[chunk['BIDTYPE'] == 'RAISE1SEC']
            
            # Update counter
            chunk_filtered_count = len(filtered_chunk)
            total_filtered_records += chunk_filtered_count
            print(f"Found {chunk_filtered_count} matching records in current chunk")
            
            # Write to output file
            if not filtered_chunk.empty:
                mode = 'w' if first_chunk else 'a'
                header = first_chunk
                filtered_chunk.to_csv(output_file, mode=mode, index=False, header=header)
                first_chunk = False
        
        print(f"Total records with BIDTYPE = RAISE1SEC: {total_filtered_records}")
        print(f"Filtered data saved to {output_file}")
        
        # Display a sample of the filtered data (read back from the saved file if it exists)
        if os.path.exists(output_file) and os.path.getsize(output_file) > 0:
            sample = pd.read_csv(output_file, nrows=5)
            print("\nSample of filtered data:")
            print(sample)
    else:
        # If BIDTYPE column doesn't exist, look for similar columns
        print("BIDTYPE column not found.")
        similar_columns = [col for col in df.columns if 'BID' in col.upper() or 'TYPE' in col.upper()]
        if similar_columns:
            print(f"Similar columns found: {similar_columns}")
            
        # Print first few rows to help understand the data structure
        print("\nFirst 3 rows of data:")
        print(df.head(3))

: 